# Verificacion de la calidad de la tabla plata.oficiales_credito

Proposito del script:  
- Verificar columna por columna la calidad de los datos.

# Estableciendo Conexion

In [1]:
# Importando las librerias y creando la conexión 
import pandas as pd 
from conexiones_y_rutas import obtener_engine
engine = obtener_engine()
df_oficiales_credito = pd.read_sql(
    "SELECT * FROM plata.oficiales_credito",
    con=engine
)

df_oficiales_tra = df_oficiales_credito.copy()

# Archivos de Ayuda

In [2]:
# Tabla limpia de sucursales, para validar, surcursal_id
df_sucursal = pd.read_sql(
    "SELECT sucursal_id, fecha_apertura FROM plata.sucursales",
    con=engine
)
df_sucursal_tra = df_sucursal.copy()
df_sucursal_tra.head()

,sucursal_id,fecha_apertura
0,1,2005-07-24
1,2,2010-01-03
2,3,2007-04-20
3,4,2010-02-19
4,5,2007-02-07


In [3]:
df_sucursal_tra["fecha_apertura"] = pd.to_datetime(df_sucursal_tra.fecha_apertura)

# Resumen de las Columnas 

- **oficial_id**: Identificador unico de cada oficial.  
- **sucursal_id**: Identificador de la sucursal asociada al oficial de credito.   
- **nombres**:  Nombres del oficial.  
- **apellido_paterno**: Apellido parterno del oficial.  
- **apellido_materno**:  Apellido materno del oficial.  
- **genero**: Genero del oficial (ejem: Masculino o Femenino).  
- **cargo**: Cargo del oficial (ejem: Analista de Creditos o Analista Senior de Credito).  
- **fecha_ingreso**: Fecha de ingreso del oficial de credito.  
- **estado**: Estado actual del oficial de credito (ejem: Activo o Inactivo)

# Verificacion de la Calidad de Datos

In [4]:
df_oficiales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   oficial_id        60 non-null     int64         
 1   sucursal_id       60 non-null     int64         
 2   nombres           60 non-null     object        
 3   apellido_paterno  60 non-null     object        
 4   apellido_materno  60 non-null     object        
 5   genero            60 non-null     object        
 6   cargo             60 non-null     object        
 7   fecha_ingreso     60 non-null     object        
 8   estado            60 non-null     object        
 9   dwh_fecha_carga   60 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(7)
memory usage: 4.8+ KB


In [5]:
df_oficiales_tra["fecha_ingreso"] = pd.to_datetime(df_oficiales_tra.fecha_ingreso)

In [6]:
df_oficiales_tra.head()

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado,dwh_fecha_carga
0,1,12,Alejandra,Herrera,Rivera,Femenino,Analista de Créditos,2017-12-20,Activo,2026-08-03 18:46:41.606666
1,2,24,Víctor,Rivera,Cusi,Masculino,Analista de Créditos,2015-06-29,Activo,2026-08-03 18:46:41.606666
2,3,8,Héctor,Vargas,López,Masculino,n/a,2010-09-20,Activo,2026-08-03 18:46:41.606666
3,4,7,Fernando,Laime,Vargas,Masculino,Analista Senior de Créditos,2020-01-01,Activo,2026-08-03 18:46:41.606666
4,5,8,Patricia,Ortiz,Ramírez,Femenino,n/a,2020-01-09,Activo,2026-08-03 18:46:41.606666


In [7]:
# Verifica si existen registro duplicados 
# Resultados Esperados: Tabla Vacia
df_oficiales_tra[df_oficiales_tra.duplicated(keep=False)]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado,dwh_fecha_carga


## oficial_id

In [8]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_oficiales_tra[df_oficiales_tra.oficial_id <= 0]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado,dwh_fecha_carga


In [9]:
# Verifica si existen ids duplicados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra[df_oficiales_tra.oficial_id.duplicated(keep=False)]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado,dwh_fecha_carga


## sucursal_id

In [10]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_oficiales_tra[df_oficiales_tra.sucursal_id <= 0]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado,dwh_fecha_carga


In [11]:
# Verifica que efectivamente los id de sucursales existan en la tabla sucursales 
# Resultados Esperados: both: 60, left_only: 0, right_only: 0
verificando_ids = df_oficiales_tra.merge(
    right=df_sucursal_tra,
    on='sucursal_id',
    how='left',
    indicator=True)
verificando_ids._merge.value_counts()

_merge
both          60
left_only      0
right_only     0
Name: count, dtype: int64

## nombres

In [12]:
# Verifica si existen nombres con formatos inadecuados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra.nombres[df_oficiales_tra.nombres != df_oficiales_tra.nombres.str.strip().str.title()]

Series([], Name: nombres, dtype: object)

## apellido_paterno

In [13]:
# Verifica si existen apellidos con formatos inadecuados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra.apellido_paterno[df_oficiales_tra.apellido_paterno != df_oficiales_tra.apellido_paterno.str.strip().str.title()]

Series([], Name: apellido_paterno, dtype: object)

## apellido_materno

In [14]:
# Verifica si existen apellidos con formatos inadecuados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra.apellido_materno[df_oficiales_tra.apellido_materno != df_oficiales_tra.apellido_materno.str.strip().str.title()]

Series([], Name: apellido_materno, dtype: object)

## genero 

In [15]:
# Resultados Esperados: "Femenino", "Masculino", "n/a"
df_oficiales_tra.genero.unique()

array(['Femenino', 'Masculino'], dtype=object)

## cargo

In [16]:
# Resultados Esperados: 'Analista de Créditos', 'n/a', 'Analista Senior de Créditos', 'Oficial de Créditos', 'Oficial Senior', 'Supervisor De Créditos'
df_oficiales_tra.cargo.unique()

array(['Analista de Créditos', 'n/a', 'Analista Senior de Créditos',
       'Oficial de Créditos', 'Oficial Senior', 'Supervisor de Créditos'],
      dtype=object)

## fecha_ingreso

In [17]:
# Muestra las fechas que generan errores al trasformar a datetime 
# Resultados Esperados: Tabla Vacia
fecha_ingreso_error = pd.to_datetime(
    df_oficiales_tra.fecha_ingreso,
    errors='coerce'
)
df_oficiales_tra.fecha_ingreso[fecha_ingreso_error.isna()]

Series([], Name: fecha_ingreso, dtype: datetime64[ns])

In [18]:
# Columnas a utilizar
df_oficiales_revi = df_oficiales_tra[['oficial_id','sucursal_id','fecha_ingreso']].copy()
df_sucursal_revi = df_sucursal_tra.copy()
# LEFT JOIN
df_merge_fechas = df_oficiales_revi.merge(
    right = df_sucursal_revi,
    how='left',
    on='sucursal_id'
)
# Verifica si existen fechas de ingreso menor a la fecha de apertura
# Resultados Esperados: Tabla Vacia
df_merge_fechas[df_merge_fechas.fecha_ingreso < df_merge_fechas.fecha_apertura].copy()

,oficial_id,sucursal_id,fecha_ingreso,fecha_apertura


## estado

In [19]:
# Resultado Esperado: 'Activo', 'Inactivo', 'n/a'
df_oficiales_tra.estado.unique()

array(['Activo', 'Inactivo'], dtype=object)